<a href="https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/milestone_4_validation_claim_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 4 — Validation and research-claim audit

This stage checks how much the evaluation result depends on the split design and records the limits of the current-label study. It compares an ordinary stratified page split with client-held-out splits. The random split is included only as a diagnostic because pages from the same client can occur on both sides. All preprocessing is fitted on training data.

The target remains `trend_direction == 'down'`. We exclude target-adjacent fields such as `trend_pct`, current-vs-prior traffic change fields, and the target text itself. The CSV has no verified observation timestamps, so this stage cannot establish future early-warning performance or causal effect.

In [1]:
# Open from the FlyRank repository, or clone it in Colab.
import os, sys, subprocess
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/flyrank-bih/flyrank-ml-internship-starter'
REPO_DIR = 'flyrank-ml-internship-starter'
CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if IN_COLAB and not CSV_PATH.exists():
    if not Path(REPO_DIR).is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
else:
    while not CSV_PATH.exists() and Path.cwd() != Path('/'):
        os.chdir('..')
        CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError('Expected data/raw/content_refresh_anonymized.csv in the FlyRank repo.')
print('CSV:', CSV_PATH.resolve())

CSV: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(CSV_PATH)
required = ['content_id','client_id','trend_direction','days_since_last_update',
            'content_age_days','impressions_90d','avg_position','ctr']
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing required fields: {missing}')
df['trend_direction'] = df['trend_direction'].astype('string').str.strip().str.lower()
df['is_declining_label'] = df['trend_direction'].map(
    {'down':1,'new':0,'up':0,'flat':0,'stable':0}
).astype('Int64')
FEATURES = ['days_since_last_update','content_age_days','impressions_90d','avg_position','ctr']
X = df[FEATURES].apply(pd.to_numeric, errors='coerce').copy()
X['avg_position'] = X['avg_position'].mask(X['avg_position'].eq(0))
X['staleness_pos_interaction'] = X['days_since_last_update'] * X['avg_position']
FEATURES = list(X.columns)
valid = (df['is_declining_label'].notna() & df['content_id'].notna() & df['client_id'].notna() &
         ~df['content_id'].duplicated(keep=False))
df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)
y = df['is_declining_label'].astype(int)
groups = df['client_id']

# Explicitly assert that known target/target-proxy columns are not model inputs.
FORBIDDEN_FEATURES = {'trend_direction','is_declining_label','trend_pct','impressions_change_pct',
                     'clicks_change_pct','current_period_impressions','previous_period_impressions'}
assert not (set(FEATURES) & FORBIDDEN_FEATURES), 'Target-adjacent feature entered the model.'
print(f'Rows: {len(df):,}; clients: {groups.nunique()}; positive rate: {y.mean():.3f}')
print('Contracted features:', FEATURES)
print('Target-adjacent fields explicitly excluded:', sorted(FORBIDDEN_FEATURES))

Rows: 30,000; clients: 32; positive rate: 0.542
Contracted features: ['days_since_last_update', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'staleness_pos_interaction']
Target-adjacent fields explicitly excluded: ['clicks_change_pct', 'current_period_impressions', 'impressions_change_pct', 'is_declining_label', 'previous_period_impressions', 'trend_direction', 'trend_pct']


## Split sensitivity

We repeat five 75/25 splits for both designs with the same fixed seeds. Stratified random splits preserve the overall class share but allow client overlap. Grouped splits hold out whole clients and better test transfer to an unseen client. For each split, logistic regression and random forest use the same training-only median imputation plus missingness indicators; average precision and Precision@50 are the main prioritization measures.

In [3]:
SEEDS = [42,43,44,45,46]
def precision_at_k(scores, labels, k=50):
    scores, labels = np.asarray(scores), np.asarray(labels)
    n = min(k, len(labels))
    return float(labels[np.argsort(-scores)[:n]].mean()) if n else np.nan

def build_model(kind, seed):
    steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if kind == 'logistic_regression':
        steps += [('scale', StandardScaler()), ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed))]
    else:
        steps += [('model', RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=2,
                   class_weight='balanced', random_state=seed, n_jobs=-1))]
    return Pipeline(steps)

rows=[]
for seed in SEEDS:
    splitters = {
        'stratified_random_page_split': next(StratifiedShuffleSplit(n_splits=1,test_size=0.25,random_state=seed).split(X,y)),
        'client_grouped_split': next(GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=seed).split(X,y,groups)),
    }
    for split_name,(train_idx,test_idx) in splitters.items():
        overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
        if split_name == 'client_grouped_split':
            assert not overlap, 'Client overlap in grouped split.'
        for model_name in ['logistic_regression','random_forest']:
            model = build_model(model_name, seed)
            model.fit(X.iloc[train_idx],y.iloc[train_idx])
            scores = model.predict_proba(X.iloc[test_idx])[:,1]
            y_test = y.iloc[test_idx]
            rows.append({
                'seed':seed,'split_design':split_name,'model':model_name,
                'train_clients':groups.iloc[train_idx].nunique(),'test_clients':groups.iloc[test_idx].nunique(),
                'overlapping_clients':len(overlap),'n_test':len(test_idx),
                'average_precision':average_precision_score(y_test,scores),
                'roc_auc':roc_auc_score(y_test,scores) if y_test.nunique()==2 else np.nan,
                'precision_at_50':precision_at_k(scores,y_test,50),
            })

split_results = pd.DataFrame(rows)
summary = (split_results.groupby(['split_design','model'])
    .agg(splits=('seed','nunique'),mean_average_precision=('average_precision','mean'),
         sd_average_precision=('average_precision','std'),mean_roc_auc=('roc_auc','mean'),
         mean_precision_at_50=('precision_at_50','mean'),sd_precision_at_50=('precision_at_50','std'),
         mean_test_rows=('n_test','mean'),mean_overlapping_clients=('overlapping_clients','mean'))
    .reset_index())
display(summary.round(3))

,split_design,model,splits,mean_average_precision,sd_average_precision,mean_roc_auc,mean_precision_at_50,sd_precision_at_50,mean_test_rows,mean_overlapping_clients
0,client_grouped_split,logistic_regression,5,0.654,0.057,0.605,0.748,0.039,9854.6,0.0
1,client_grouped_split,random_forest,5,0.663,0.046,0.611,0.708,0.058,9854.6,0.0
2,stratified_random_page_split,logistic_regression,5,0.656,0.007,0.657,0.752,0.083,7500.0,31.4
3,stratified_random_page_split,random_forest,5,0.728,0.003,0.720,0.868,0.033,7500.0,31.4


## Leakage and claim audit

The feature contract excludes the supplied label, its direction text, `trend_pct`, and explicit current-vs-prior traffic changes. Imputation and scaling occur inside each training fold. Client overlap is measured and asserted absent for grouped splits. These checks address target and client leakage risks, but they cannot establish that the remaining metrics were captured before the label window: that requires dated warehouse records.

Use the grouped results for the main claim. A safe report statement is: *In this anonymized snapshot, the model ranked pages carrying the supplied decline label under client-held-out evaluation; results are exploratory and do not show that the model predicts future declines.* Do not write a general claim of prevention, causality, or early warning.

In [4]:
OUTPUT_DIR = Path('work/outputs/milestone_4_validation_claim_audit')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
split_results.to_csv(OUTPUT_DIR/'split_sensitivity_results.csv',index=False)
summary.to_csv(OUTPUT_DIR/'split_sensitivity_summary.csv',index=False)
contract = {
    'target':'trend_direction == down; current supplied label',
    'unit':'one unique content_id within client_id',
    'features':FEATURES,
    'excluded_target_adjacent_fields':sorted(FORBIDDEN_FEATURES),
    'preprocessing':'median imputation with missing indicators inside training-only pipeline',
    'validation':'five stratified random page splits as a diagnostic; five client-grouped splits as primary',
    'primary_claim':'current-label prioritization on held-out clients only',
    'limitations':['random split permits client overlap','only 32 clients','CSV feature dates unverified','not a future-warning or causal evaluation'],
}
(OUTPUT_DIR/'validation_claim_contract.json').write_text(json.dumps(contract,indent=2)+'\n',encoding='utf-8')
print('Saved validation audit to',OUTPUT_DIR.resolve())

Saved validation audit to /content/flyrank-ml-internship-starter/work/outputs/milestone_4_validation_claim_audit
